# IC-1102 — Case study: two categories, then three

**ML & FinTech · 115-1 · 20261102 · Session 3 · work in pairs · 40 minutes, then discussion**

| Part | Min | What you do |
|---|---|---|
| Case 1 | 15 | Two categories: the model versus "approve everyone", then a cutoff for your desk's costs |
| Case 2 | 15 | Three categories: multinomial logistic regression, then read its results and coefficients |
| Hand in | 10 | Two agreed sentences on **one sheet of paper per pair** |

You may use any AI tool for the code. Hand in the paper sheet and you get full credit for today's exercise; be ready to explain
your numbers. Your numbers come from your 學號; your partner's differ.

---

## Case 1: two categories

The bank's dashboard reports the default model is **82% accurate**. Your manager says:

> *"We also found that simply **approving everyone** is 78% accurate. Not far off, so no model needed.
> And if we do keep the model, the cutoff is 0.5 — that is what 'probability of default' means."*

Each desk is different: on **your** desk a missed default costs **c** times as much as a refused
good customer (your c is printed in the setup). Your manager asks: **"Model or no model? And if the
model, which cutoff?"**

## Case 2: three categories

The bank now wants three groups, not two: **1 = paid on time, 2 = late, 3 = default**. Your manager says:

> *"The three-category model is about 73% accurate, so it works for all three groups. And the `BILL_1` number
> in the default row of the softmax output is the effect of the bill on default."*

Is your manager right on either point?


## Setup

In [ ]:
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings("ignore")

In [ ]:
STUDENT_ID = ""    # your 學號, in quotes, e.g. "315707999"
PARTNER_ID = ""    # your partner's 學號

In [ ]:
URL  = ("https://raw.githubusercontent.com/202609-ML-FinTech/00-course-info/main/slides/"
        "default%20of%20credit%20card%20clients.csv")
CAND = ["default of credit card clients.csv", "../slides/default of credit card clients.csv",
        "../00-course-info/slides/default of credit card clients.csv"]
path = next((p for p in CAND if Path(p).exists()), URL)
credit = pd.read_csv(path).rename(columns={"X1":"LIMIT","X5":"AGE","X6":"PAY_0","X7":"PAY_2",
                                           "X12":"BILL_1","X18":"PAY_AMT_1","Y":"default"})
credit["LIMIT"] = credit["LIMIT"] / 10000          # NT$ ten-thousands
credit["BILL_1"] = credit["BILL_1"] / 10000
credit["PAY_AMT_1"] = credit["PAY_AMT_1"] / 10000
credit = credit[credit["PAY_0"] <= 8]                # one record has PAY_0 = 3032: a data error (IC-02). Dropped.
# 30,000 Taiwanese credit-card clients (2005). default = 1 if the client defaulted next month.
# PAY_0: months late on the most recent payment (-1 = paid on time, 0 = revolving, 1..9 months late).

def my_sample(student_id, n=3000):
    """Your desk's 3,000 applicants. The same 學號 always gives the same applicants."""
    seed = int(hashlib.sha256(f"{int(student_id)}-credit".encode()).hexdigest(), 16) % 2**32
    rng = np.random.default_rng(seed)
    return credit.iloc[np.sort(rng.choice(len(credit), n, replace=False))].copy()

assert str(STUDENT_ID).strip().isdigit(), "Fill in STUDENT_ID in the cell above, then run again."
desk = my_sample(STUDENT_ID)
print(f"Desk of {STUDENT_ID}: {len(desk)} applicants, {desk['default'].mean()*100:.1f}% defaulted")
MY_COST = 3 + int(hashlib.sha256(f"{int(STUDENT_ID)}-cost".encode()).hexdigest(), 16) % 6
print(f"On my desk a missed default costs {MY_COST} times a refused good customer.")

---
# Case 1, part 1 — Model or no model? (7 min)

Split (given), fit a logistic regression of `default` on `LIMIT, PAY_0, PAY_2, BILL_1, PAY_AMT_1`
on `train` (use `LogisticRegression(max_iter=3000)`), and get `p = predict_proba(test)[:, 1]`. Then print, on `test`:
accuracy of "approve everyone", and accuracy, sensitivity, specificity of the model at cutoff 0.5.


In [ ]:
from sklearn.model_selection import train_test_split
seed = int(hashlib.sha256(f"{int(STUDENT_ID)}-split".encode()).hexdigest(), 16) % 2**32
train, test = train_test_split(desk, test_size=0.4, random_state=seed, stratify=desk["default"])
print(len(train), "to fit,", len(test), "to test")

In [ ]:
# 2: fit; p on test; the two accuracies, sensitivity and specificity at 0.5



**Q1.** My desk: approve-everyone accuracy ____ ; model at 0.5: accuracy ____ , sensitivity ____ , specificity ____ .

**Q2.** Is the manager right that the two are "the same"? What does accuracy hide here?

> _replace this line_

---
# Case 1, part 2 — The cutoff for your desk (8 min)

For cutoffs 0.1, 0.2, 0.3, 0.4, 0.5 compute on `test`:
expected cost = (false positives × 1) + (false negatives × MY_COST). Print the table and pick the cheapest.


In [ ]:
# 3: cost at each cutoff



**Q3.** My c = ____ ; cheapest cutoff = ____ ; at that cutoff sensitivity = ____ and specificity = ____ .

**Q4.** Swap with your partner.

| | c | best cutoff | sensitivity there |
|---|---|---|---|
| me | | | |
| my partner | | | |

Same model, different cutoffs — why? Is 0.5 special at all?

> _replace this line_

---
# Case 2 — Three categories (15 min)

The classes and the fit are given. `PAY_0` defines class 2, so it is **not** a predictor here.


In [ ]:
def status(d):
    # 3 = default next month; otherwise 2 = late on the latest payment (PAY_0 >= 1); otherwise 1 = paid on time
    return np.where(d["default"] == 1, 3, np.where(d["PAY_0"] >= 1, 2, 1))
X3 = ["LIMIT", "PAY_2", "BILL_1", "PAY_AMT_1"]
y3_train, y3_test = status(train), status(test)
from sklearn.linear_model import LogisticRegression
m3 = LogisticRegression(max_iter=5000).fit(train[X3], y3_train)    # three classes: multinomial (softmax)
pred3 = m3.predict(test[X3])                                       # the class with the largest probability
softmax = pd.DataFrame(m3.coef_, index=["class 1", "class 2", "class 3"], columns=X3)
softmax.round(3)


In [ ]:
# 4: (a) accuracy of pred3, and accuracy of always predicting class 1
#    (b) for each class k: of the clients truly in class k, the share predicted as k
#    (c) baseline coding with class 1 as baseline: subtract the class 1 row from every row




**Q5.** Accuracy ____ ; always predicting class 1 ____ ; share caught: class 1 ____ , class 2 ____ , class 3 ____ .

**Q6.** `BILL_1`, default row: softmax ____ ; baseline (against class 1) ____ . What does the baseline number say? Compare with your partner.

> _replace this line_


---
# Hand in, on paper (10 min)

**One sheet per pair**: both names and 學號, the two tables for **each** of you, and **two sentences, agreed between you**, one per case.

Give the sheet to Prof. Teng or Claire before you leave.

Then: *Restart & Run All*, and push this notebook as `in-class-exercise/1102/IC-1102.ipynb`
in your own repository before **12:10**.
